# MERRA-2 Daily Weather for LA County (2016–2025)

Turns 3,652 hourly NetCDF granules into one daily weather table, parallel to the
daily AQI series built in `aqi_pipeline.ipynb`.

Each granule is a single day: **24 hourly time steps × a 3×3 grid** covering the
LA bounding box (33.5–34.5 °N, 118.75–117.5 °W), with four variables from the
M2T1NXSLV product:

| variable | meaning | units |
|---|---|---|
| `T2M`  | 2-meter air temperature   | K |
| `QV2M` | 2-meter specific humidity | kg/kg |
| `U10M` | 10-meter eastward wind    | m/s |
| `V10M` | 10-meter northward wind   | m/s |

Output: `data/processed/la_daily_weather_2016_2025.csv`.

## Setup

Paths are anchored to the repo root so this runs from a fresh clone.

In [1]:
import glob, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA      = ROOT / 'data'
GRANULES  = DATA / 'raw' / 'merra2_slv'
PROCESSED = DATA / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

OUT_CSV = PROCESSED / 'la_daily_weather_2016_2025.csv'

assert GRANULES.exists(), (
    f'No MERRA-2 granules at {GRANULES}. They are gitignored (~200 MB); '
    're-download with data/raw/merra2_manifest.txt — see README.'
)

files = sorted(glob.glob(str(GRANULES / '*.nc4')))
print(f'{len(files)} granules found')
print('first:', Path(files[0]).name)
print('last: ', Path(files[-1]).name)

3652 granules found
first: M2T1NXSLV.5.12.4:MERRA2_400.tavg1_2d_slv_Nx.20160101.nc4
last:  M2T1NXSLV.5.12.4:MERRA2_401.tavg1_2d_slv_Nx.20210930.nc4


## What one granule looks like

Before processing 3,652 files, confirm the shape and grid of a single one.

In [2]:
ds = xr.open_dataset(files[0])
print(ds.dims)
print('lat:', ds.lat.values)
print('lon:', ds.lon.values)
print('time steps:', len(ds.time), '| first:', str(ds.time.values[0]), '| last:', str(ds.time.values[-1]))
print()
for v in ['T2M', 'QV2M', 'U10M', 'V10M']:
    print(f'{v:5} {ds[v].attrs.get("units","?"):8} {ds[v].attrs.get("long_name","?")}')
ds.close()

FrozenMappingWarningOnValuesAccess({'time': 24, 'lat': 3, 'lon': 3})
lat: [33.5 34.  34.5]
lon: [-118.75  -118.125 -117.5  ]
time steps: 24 | first: 2016-01-01T00:30:00.000000000 | last: 2016-01-01T23:30:00.000000000

T2M   K        2-meter_air_temperature
QV2M  kg kg-1  2-meter_specific_humidity
U10M  m s-1    10-meter_eastward_wind
V10M  m s-1    10-meter_northward_wind


Nine grid cells, hourly time stamps centered on the half hour (00:30 … 23:30).
The 3×3 box straddles the coastline, so the southwest cell is largely ocean — we
take a spatial mean across all nine, which is the standard way to get a single
regional value and matches how the AQI series pools all monitors into one
city-wide number.

## Aggregating hourly → daily

For each granule: average the 3×3 grid down to one value per hour, then reduce
the 24 hours to daily statistics.

One ordering detail matters. **Wind speed is computed per hour first, then
averaged** — not derived from the daily mean of U and V. A day that blows hard
east in the morning and hard west in the evening has a high mean speed but a
near-zero mean vector, and the second method would wrongly call it calm.
Direction is the opposite case: it's a vector quantity, so the daily direction
comes from the mean U/V components.

`calm_hours` counts hours below 2 m/s — a stagnation proxy, since trapped air is
what drives LA's worst smog days.

In [3]:
def wind_direction(u, v):
    """Meteorological wind direction in degrees: the direction wind blows FROM.
    0 = from the north, 90 = from the east."""
    return np.degrees(np.arctan2(-u, -v)) % 360


def summarize_granule(path):
    """One granule (24 h x 3 x 3) -> one dict of daily values."""
    with xr.open_dataset(path) as ds:
        # spatial mean across the 9 cells -> one value per hour
        t2m  = ds.T2M.mean(dim=('lat', 'lon')).values - 273.15   # K -> degrees C
        qv2m = ds.QV2M.mean(dim=('lat', 'lon')).values * 1000.0  # kg/kg -> g/kg
        u    = ds.U10M.mean(dim=('lat', 'lon')).values
        v    = ds.V10M.mean(dim=('lat', 'lon')).values
        date = pd.Timestamp(ds.time.values[0]).normalize()

    speed = np.sqrt(u**2 + v**2)   # hourly speed, THEN averaged
    u_bar, v_bar = u.mean(), v.mean()

    return {
        'date':             date,
        't2m_mean':         t2m.mean(),
        't2m_max':          t2m.max(),
        't2m_min':          t2m.min(),
        't2m_range':        t2m.max() - t2m.min(),
        'qv2m_mean':        qv2m.mean(),
        'qv2m_max':         qv2m.max(),
        'u10m_mean':        u_bar,
        'v10m_mean':        v_bar,
        'wind_speed_mean':  speed.mean(),
        'wind_speed_max':   speed.max(),
        'wind_speed_min':   speed.min(),
        'wind_dir_mean':    wind_direction(u_bar, v_bar),
        'calm_hours':       int((speed < 2.0).sum()),
    }

In [4]:
rows, failures = [], []
t0 = time.time()
CHECKPOINT = 500

for i, f in enumerate(files, 1):
    try:
        rows.append(summarize_granule(f))
    except Exception as e:
        failures.append((Path(f).name, repr(e)))   # log and keep going

    if i % CHECKPOINT == 0 or i == len(files):
        elapsed = time.time() - t0
        eta = elapsed / i * (len(files) - i)
        print(f'{i:5}/{len(files)}  {elapsed:5.1f}s elapsed  ~{eta:4.1f}s remaining  '
              f'({len(failures)} failures)')

weather = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
print(f'\nBuilt {len(weather)} daily rows in {time.time()-t0:.1f}s')

if failures:
    print(f'\n{len(failures)} granules failed:')
    for name, err in failures[:10]:
        print(' ', name, '->', err)
else:
    print('No failures.')

  500/3652    3.0s elapsed  ~18.8s remaining  (0 failures)


 1000/3652    5.8s elapsed  ~15.4s remaining  (0 failures)


 1500/3652    8.6s elapsed  ~12.4s remaining  (0 failures)


 2000/3652   11.5s elapsed  ~ 9.5s remaining  (0 failures)


 2500/3652   14.4s elapsed  ~ 6.6s remaining  (0 failures)


 3000/3652   17.2s elapsed  ~ 3.7s remaining  (0 failures)


 3500/3652   20.1s elapsed  ~ 0.9s remaining  (0 failures)


 3652/3652   21.0s elapsed  ~ 0.0s remaining  (0 failures)

Built 3652 daily rows in 21.0s
No failures.


## Sanity checks

Three things to verify before this feeds a model: no missing days, no duplicate
days, and physically plausible value ranges.

In [5]:
print(f'rows: {len(weather)}')
print(f'range: {weather.date.min().date()} -> {weather.date.max().date()}')
print(f'duplicate dates: {weather.date.duplicated().sum()}')

full = pd.date_range(weather.date.min(), weather.date.max(), freq='D')
missing = full.difference(weather.date)
print(f'gaps in the date range: {len(missing)}', list(missing[:5]))

print(f'\nNaNs per column:\n{weather.isna().sum().to_string()}')
print(f'\n{weather.describe().T.to_string()}')

rows: 3652
range: 2016-01-01 -> 2025-12-30
duplicate dates: 0
gaps in the date range: 0 []

NaNs per column:
date               0
t2m_mean           0
t2m_max            0
t2m_min            0
t2m_range          0
qv2m_mean          0
qv2m_max           0
u10m_mean          0
v10m_mean          0
wind_speed_mean    0
wind_speed_max     0
wind_speed_min     0
wind_dir_mean      0
calm_hours         0

                  count                           mean                  min                  25%                  50%                  75%                  max        std
date               3652  2020-12-30 12:00:00.000000256  2016-01-01 00:00:00  2018-07-01 18:00:00  2020-12-30 12:00:00  2023-07-01 06:00:00  2025-12-30 00:00:00        NaN
t2m_mean         3652.0                      17.247824             5.753351            12.971289            16.817544            21.474236            32.940414   5.260856
t2m_max          3652.0                      23.011406             7.419067        

## Does it look like Los Angeles?

A quick seasonal check. If the aggregation is right, temperature should peak in
late summer and wind should be strongest in spring.

In [6]:
seasonal = (weather
            .assign(month=weather.date.dt.month)
            .groupby('month')[['t2m_mean', 't2m_max', 'qv2m_mean',
                               'wind_speed_mean', 'calm_hours']]
            .mean().round(2))
print(seasonal.to_string())

        t2m_mean    t2m_max  qv2m_mean  wind_speed_mean  calm_hours
month                                                              
1      11.440000  16.590000       5.80             3.17        7.23
2      11.920000  17.309999       5.81             3.19        7.41
3      12.700000  17.950001       6.48             3.21        7.00
4      15.200000  21.040001       7.00             2.99        8.01
5      16.780001  22.610001       7.80             2.82        8.40
6      20.629999  26.879999       8.28             2.55       10.38
7      23.709999  29.809999       8.74             2.38       11.82
8      24.219999  30.389999       9.00             2.31       12.26
9      22.580000  28.549999       9.14             2.33       11.83
10     19.520000  25.680000       7.45             2.35       11.75
11     15.480000  21.260000       6.17             2.66        9.55
12     12.480000  17.740000       5.75             2.87        8.80


August is the hottest month (24.2 °C mean) and humidity peaks just after it in
September — the late-summer monsoon signature. Wind runs the other way: strongest
in late winter and early spring (~3.2 m/s, Jan–Mar) and weakest in midsummer
(~2.3 m/s), when calm hours nearly double, to roughly half the day below 2 m/s.

That hot-and-stagnant summer combination is precisely LA's ozone season, so these
features should carry real signal for the model.

## Saving the daily weather dataset

In [7]:
weather.to_csv(OUT_CSV, index=False)
print(f'Saved {OUT_CSV.relative_to(ROOT)}  ({len(weather)} rows, {weather.shape[1]} columns)')
weather.head()

Saved data/processed/la_daily_weather_2016_2025.csv  (3652 rows, 14 columns)


,date,t2m_mean,t2m_max,t2m_min,t2m_range,qv2m_mean,qv2m_max,u10m_mean,v10m_mean,wind_speed_mean,wind_speed_max,wind_speed_min,wind_dir_mean,calm_hours
0,2016-01-01,7.802708,14.086761,4.433899,9.652863,2.930061,3.288353,-3.511768,-2.129953,4.326950,6.723966,0.835390,58.762402,4
1,2016-01-02,9.579383,15.884277,6.586517,9.297760,3.442668,5.004436,-2.350438,-0.510819,2.998282,4.193285,0.585935,77.738617,2
2,2016-01-03,10.133282,14.810822,7.110870,7.699951,5.310264,6.765719,-2.396616,1.754932,3.313107,6.091429,1.075208,126.213608,4
3,2016-01-04,11.156650,14.856079,8.998993,5.857086,6.437552,7.229472,-3.336517,0.726196,3.842359,5.233736,1.065249,102.278969,4
4,2016-01-05,11.257178,13.084229,10.060364,3.023865,7.811948,8.766274,0.448972,3.415153,4.437946,9.754294,1.011675,187.489426,9
